Scenario: Legal Research Assistant for a Corporate Compliance Team
Context

A corporate compliance department constantly reviews lengthy legal documents, regulatory filings, and policy updates. These documents are dense, full of
legal terminology, and often hundreds of pages long. The team struggles to quickly extract relevant clauses or understand implications without spending hours reading.

How the RAG Chatbot Fits In
- Input Source: The team uploads a legal document (e.g., data_privacy_regulation.pdf).
- Chunking: The chatbot splits the document into sections (clauses, articles, sub-sections) so no detail is overlooked.
- Embeddings + Vector DB: Each section is converted into embeddings and stored in Chroma, enabling semantic search rather than keyword-only lookup.
- Retriever: When someone asks, “What does this regulation say about cross-border data transfers?”, the retriever surfaces the most relevant clauses.
- LLM Response: A Hugging Face model (e.g., Flan-T5) generates a concise, plain-language summary of those clauses, stripping away heavy legal jargon.
- Chat Loop: The compliance team can continue asking questions interactively, like “Does this regulation conflict with GDPR?” or “What penalties are mentioned for non-compliance?”.

Outcome

The chatbot acts as a legal research assistant, helping the compliance team quickly interpret complex documents, identify risks, and prepare summaries for executives
without needing to manually parse every page.

I will be using deepset/robert-base-squad2 model in this project


In [ ]:
!pip install chromadb

In [ ]:
!pip install sentence-transformers pypdf transformers

In [5]:
import os
from transformers import pipeline
import chromadb
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

Step 2: Load the document

In [6]:
reader=PdfReader('/content/data_privacy_regulation.pdf')
text=' '
for page in reader.pages:
  text+=page.extract_text()

print('document Loaded')

document Loaded


Step 3: Chunk the document

In [13]:
print('Splitting document into chunks...')

def chunk_text(text,chunk_size=400,overlap=80):
  chunks=[]
  start=0

  while start<len(text):
    end=start+chunk_size
    chunk=text[start:end]
    chunks.append(chunk)
    start+=chunk_size-overlap

  return chunks
chunks=chunk_text(text)
print("Total chunks Created: ",len(chunks))

Splitting document into chunks...
Total chunks Created:  10


Step 4: Create embedding

In [15]:
embedding_model= SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print('Embedding Model Loaded')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Model Loaded


Step 5: Create vector database

In [16]:
client=chromadb.Client()
try:
  client.delete_collection('pdf_collection')
except:
  pass

collection=client.create_collection('pdf_collection')
print('New vector collection created')

New vector collection created


Step 6: Store chunks in database

In [17]:
for i, chunk in enumerate(chunks):
  embedding=embedding_model.encode(chunk).tolist()

  collection.add(documents=[chunk],
                 embeddings=[embedding],
                 ids=[str(i)])

print('All chunks are stored successfully')

All chunks are stored successfully


Step 7: Retriever Function (converts user questions -> embedding)

In [18]:
def retrieve(query,k=3):
  query_embedding=embedding_model.encode(query).tolist()

  results=collection.query(query_embeddings=[query_embedding],
                           n_results=k)

  return results['documents'][0]

Step 8: Load the LLM

In [19]:
qa_pipeline=pipeline("question-answering",
    model="deepset/roberta-base-squad2"
)

print('LLM loaded successfully')

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

LLM loaded successfully


Step 9: Question Answering Function

In [25]:
def answer_question(query):
  context_docs=retrieve(query)
  print("Retrieved docs:", context_docs)
  context=' '.join(context_docs)

  prompt=f'''You are an AI assistant.
  Answer the question using ONLY the context below. If the answer is not present,
  "Not found in document".

  Context:
  {context}

  Question:
  {query}

  Answer:
  '''
  response = qa_pipeline(
    question=query,
    context=context
)
  print("RAW MODEL OUTPUT:", response)
  return response['answer']

Step 10: Chat Loop

In [26]:
print("RAG Chatbot Ready")
print("Type 'exit' to stop")


while True:
  question = input('Ask a question: ')
  if question.lower()=='exit':
    print('Goodbye!')
    break

  answer=answer_question(question)

  print('\nAnswer:\n',answer)
  print('\n'+'-'*60+'\n')

RAG Chatbot Ready
Type 'exit' to stop
Ask a question: Does this regulation conflict with GDPR?
Retrieved docs: ['y.\nRepeated violations may result in suspension of data processing activities.\nArticle 7: Compliance and Audits\nOrganizations must maintain documentation demonstrating compliance with this regulation.\nRegulatory authorities may conduct periodic audits to ensure adherence to data protection\nprinciples and security standards.\n', ' Data Privacy and Protection Regulation (Sample\n Document)\nThis sample regulation is created for educational and demonstration purposes. It simulates a legal\ndocument that can be used for testing Retrieval-Augmented Generation (RAG) systems. The\nregulation outlines rules related to data collection, storage, processing, cross-border transfers, and\npenalties for non■compliance.\nArticle 1: Definition', " 6: Penalties for Non■Compliance\nOrganizations found to be in violation of this regulation may face financial penalties, regulatory\ninvesti